# Teste do streaming em um único notebook (producer + consumer)

O Colab gratuito não deixa abrir duas sessões ao mesmo tempo. Aqui o **consumer roda em uma thread** (fica escutando) enquanto o **producer publica**, na mesma sessão. Rode as células em ordem; a última faz o teste.

## 1 — instalação

In [ ]:
!pip install -q google-cloud-bigquery google-cloud-pubsub

## 2 — login Google

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    print('Fora do Colab: usando as credenciais locais.')

## 3 — configuração do consumer

In [ ]:
import json
import time
from datetime import datetime, timezone

from google.api_core.exceptions import DeadlineExceeded   # NOVO: o pull termina assim quando a fila está vazia
from google.cloud import bigquery, pubsub_v1

project_id = "project-ce782257-1c9c-4b6f-87a"
# CORREÇÃO: o código usava "sub-alfabetizacao", que não existe; a assinatura real é esta
subscription_id = "topico-alfabetizacao-sub"
# CORREÇÃO: evento novo entra na BRONZE (só acrescenta). A silver_tc.alunos é recriada inteira
# a cada execução da frente 3, então tudo que fosse gravado lá seria apagado.
tabela_eventos = f"{project_id}.bronze_tc.eventos_alunos"
# Eventos ruins ficam guardados aqui, sem derrubar o pipeline.
tabela_rejeitados = f"{project_id}.ops_tc.eventos_rejeitados"

subscriber = pubsub_v1.SubscriberClient()
subscription_path = subscriber.subscription_path(project_id, subscription_id)
bq_client = bigquery.Client(project=project_id)

# Guarda os id_evento já vistos nesta execução (dedup em memória).
# Zera se o notebook reiniciar; a Silver (frente 3) deduplica de novo por id_evento.
eventos_processados = set()

# NOVO: a assinatura tem prazo de confirmação (ack) de só 10 s e cada gravação no BigQuery leva alguns
# segundos. Estourado o prazo, o Pub/Sub reentrega a mensagem. Pedimos mais tempo logo após ler o lote.
PRAZO_ACK_SEGUNDOS = 60

## 4 — schemas

In [ ]:
# Schema fixo = a tabela sempre nasce com os tipos certos (caderno texto, avaliado booleano etc.),
# em vez de o BigQuery adivinhar pelo primeiro lote. Mesmos nomes e tipos da silver_tc.alunos,
# mais as colunas do streaming (id_evento, origem, horários e latência).
SCHEMA_EVENTOS = [
    bigquery.SchemaField("id_evento", "STRING"),
    bigquery.SchemaField("origem", "STRING"),
    bigquery.SchemaField("horario_producao", "TIMESTAMP"),
    bigquery.SchemaField("horario_recebimento", "TIMESTAMP"),
    bigquery.SchemaField("latencia_segundos", "FLOAT64"),
    bigquery.SchemaField("ano", "INT64"),
    bigquery.SchemaField("id_municipio", "STRING"),
    bigquery.SchemaField("cod_uf", "STRING"),
    bigquery.SchemaField("sigla_uf", "STRING"),
    bigquery.SchemaField("id_escola", "STRING"),
    bigquery.SchemaField("id_aluno", "STRING"),
    bigquery.SchemaField("caderno", "STRING"),
    bigquery.SchemaField("serie", "STRING"),
    bigquery.SchemaField("serie_descricao", "STRING"),
    bigquery.SchemaField("rede", "STRING"),
    bigquery.SchemaField("rede_descricao", "STRING"),
    bigquery.SchemaField("presenca", "STRING"),
    bigquery.SchemaField("presenca_descricao", "STRING"),
    bigquery.SchemaField("preenchimento_caderno", "STRING"),
    bigquery.SchemaField("preenchimento_caderno_descricao", "STRING"),
    bigquery.SchemaField("alfabetizado", "STRING"),
    bigquery.SchemaField("alfabetizado_descricao", "STRING"),
    bigquery.SchemaField("avaliado", "BOOL"),
    bigquery.SchemaField("proficiencia", "FLOAT64"),
    bigquery.SchemaField("peso_aluno", "FLOAT64"),
]
# Lista só com os nomes: usada para montar cada linha apenas com as colunas do schema
# (campo extra no evento não quebra o load job).
CAMPOS_EVENTOS = [c.name for c in SCHEMA_EVENTOS]

# Tabela de rejeitados com 4 colunas fixas, sempre as mesmas, não importa o que veio quebrado.
# O evento original inteiro fica em `mensagem_bruta` (texto), então nada se perde
# e o schema não muda entre lotes (evento sem campos ou JSON ilegível também cabe aqui).
SCHEMA_REJEITADOS = [
    bigquery.SchemaField("id_evento", "STRING"),          # pode ser nulo (JSON quebrado não tem id)
    bigquery.SchemaField("horario_recebimento", "TIMESTAMP"),
    bigquery.SchemaField("motivo_rejeicao", "STRING"),    # por que foi rejeitado
    bigquery.SchemaField("mensagem_bruta", "STRING"),     # a mensagem original, sem alterações
]

## 5 — validação e gravação

In [ ]:
LIMIAR_ALFABETIZADO = 743   # mesmo limiar do INEP usado no producer
# Campos sem os quais o evento não serve para a Silver/Gold.
OBRIGATORIOS = ["id_evento", "horario_producao", "ano", "id_municipio",
                "id_aluno", "proficiencia", "peso_aluno", "alfabetizado"]


def agora_utc():
    # Sempre com fuso UTC, para subtrair do horário do producer (que também tem fuso).
    return datetime.now(timezone.utc)


def motivo_invalido(dados):
    """Valida o evento. Devolve o MOTIVO da rejeição (texto) ou None se estiver válido.
    CORREÇÃO: a validação antiga conferia `codigo_municipio` e `taxa_alfabetizacao`, campos que o
    producer não envia, então todo evento era rejeitado. Aqui conferimos os campos reais."""
    # `is None` (e não `not`): "0" ou 0 são valores legítimos e não podem contar como ausentes.
    faltando = [c for c in OBRIGATORIOS if dados.get(c) is None]
    if faltando:
        return f"campos ausentes: {faltando}"

    # município deve ter 7 dígitos (padrão IBGE), como nas demais tabelas
    if len(str(dados["id_municipio"])) != 7:
        return "id_municipio fora do padrão IBGE (7 dígitos)"

    # proficiência e peso precisam ser números (texto como "abc" cai aqui)
    try:
        proficiencia = float(dados["proficiencia"])
        peso = float(dados["peso_aluno"])
    except (TypeError, ValueError):
        return "proficiência ou peso não numérico"
    if not (0 < proficiencia < 1000):                  # a escala real vai de ~578 a ~904
        return "proficiência fora da faixa"
    if peso <= 0:                                      # na fonte nenhum peso é <= 0
        return "peso_aluno menor ou igual a zero"

    # consistência: alfabetizado tem que bater com a proficiência (743). Evita o "Sim" com 600 pontos.
    esperado = "1" if proficiencia >= LIMIAR_ALFABETIZADO else "0"
    if str(dados["alfabetizado"]) != esperado:
        return f"alfabetizado inconsistente com a proficiência (esperado {esperado})"

    # o horário de produção precisa ser legível, senão não dá para medir a latência
    try:
        datetime.fromisoformat(str(dados["horario_producao"]))
    except ValueError:
        return "horario_producao inválido"
    return None


def montar_rejeitado(texto_bruto, motivo, id_evento=None):
    """Monta a linha de ops_tc.eventos_rejeitados (as 4 colunas do SCHEMA_REJEITADOS)."""
    return {
        "id_evento": id_evento,
        "horario_recebimento": agora_utc().isoformat(),
        "motivo_rejeicao": motivo,
        "mensagem_bruta": texto_bruto,
    }


def gravar(linhas, tabela, schema):
    """Grava por LOAD JOB em modo append.
    CORREÇÃO: o código antigo usava `insert_rows_json` (streaming insert), que o sandbox bloqueia e
    que NÃO levanta erro quando falha (devolve uma lista de erros que era ignorada), então o notebook
    dizia "inseridos" sem ter gravado nada. O load job é o mesmo método da Bronze e funciona no sandbox.
    `job.result()` espera terminar e levanta exceção se falhar, de propósito: o ack só ocorre se der certo."""
    config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND", schema=schema)
    job = bq_client.load_table_from_json(linhas, tabela, job_config=config)
    job.result()
    print(f"{len(linhas)} linhas gravadas em {tabela} (job {job.job_id})")

## 6 — consumo em micro-lote

In [ ]:
def consumir_micro_lotes(max_mensagens=10, espera_segundos=15):
    """Lê até `max_mensagens`, valida, grava e só então confirma (ack). Devolve quantas mensagens leu
    (0 se a fila estava vazia). `espera_segundos`: quanto tempo esperar por mensagens antes de desistir."""
    try:
        response = subscriber.pull(request={"subscription": subscription_path, "max_messages": max_mensagens},
                                   timeout=espera_segundos)
    except DeadlineExceeded:          # NOVO: fila vazia durante a espera; não é erro
        return 0
    if not response.received_messages:
        return 0

    # NOVO: estende o prazo de confirmação ANTES de gravar, para a gravação não estourar os 10 s da
    # assinatura e provocar reentrega da mesma mensagem.
    ack_ids = [m.ack_id for m in response.received_messages]
    subscriber.modify_ack_deadline(request={"subscription": subscription_path, "ack_ids": ack_ids,
                                            "ack_deadline_seconds": PRAZO_ACK_SEGUNDOS})

    linhas_validas, linhas_invalidas = [], []
    duplicados = 0

    for msg in response.received_messages:
        # "replace" troca bytes ilegíveis por "?" em vez de levantar erro de decodificação
        texto = msg.message.data.decode("utf-8", "replace")

        # CORREÇÃO: antes o json.loads não tinha proteção, e uma mensagem quebrada derrubava o consumer
        try:
            dados = json.loads(texto)
        except json.JSONDecodeError:
            linhas_invalidas.append(montar_rejeitado(texto, "JSON inválido"))
            continue
        # JSON válido, mas que não é um objeto (ex.: uma lista ou número): também é rejeitado
        if not isinstance(dados, dict):
            linhas_invalidas.append(montar_rejeitado(texto, "mensagem não é um objeto JSON"))
            continue

        # Duplicado: o mesmo id_evento já foi aceito. É só ignorado (será confirmado no fim).
        # A checagem vem ANTES da validação, mas o id só entra no set depois de validado:
        # assim, um inválido reenviado é rejeitado de novo e não some como "duplicado".
        id_evento = dados.get("id_evento")
        if id_evento is not None and id_evento in eventos_processados:
            duplicados += 1
            continue

        motivo = motivo_invalido(dados)
        if motivo:
            linhas_invalidas.append(montar_rejeitado(texto, motivo, id_evento))
            continue

        eventos_processados.add(id_evento)

        # Latência = recebimento menos produção, ambos em UTC com fuso.
        recebimento = agora_utc()
        producao = datetime.fromisoformat(str(dados["horario_producao"]))
        if producao.tzinfo is None:                    # tolera evento antigo sem fuso, assumindo UTC
            producao = producao.replace(tzinfo=timezone.utc)
        dados["horario_recebimento"] = recebimento.isoformat()
        dados["latencia_segundos"] = (recebimento - producao).total_seconds()

        # Fica só com as colunas do schema; o que não está nele (campo extra) é descartado
        linhas_validas.append({c: dados.get(c) for c in CAMPOS_EVENTOS})

    # GRAVAR PRIMEIRO, CONFIRMAR DEPOIS.
    # CORREÇÃO: antes o msg.ack() vinha antes de gravar; se o BigQuery falhasse, a mensagem já estava
    # confirmada e se perdia. Agora, se `gravar` levantar erro, o ack não acontece e o Pub/Sub reentrega.
    if linhas_validas:
        gravar(linhas_validas, tabela_eventos, SCHEMA_EVENTOS)
    if linhas_invalidas:
        gravar(linhas_invalidas, tabela_rejeitados, SCHEMA_REJEITADOS)

    # Confirma TODAS as mensagens do lote (válidas, duplicadas e rejeitadas): as rejeitadas já estão
    # guardadas e as duplicadas não precisam ser reprocessadas.
    subscriber.acknowledge(request={"subscription": subscription_path, "ack_ids": ack_ids})

    print(f"Lote: {len(response.received_messages)} recebidas | {len(linhas_validas)} válidas | "
          f"{duplicados} duplicadas ignoradas | {len(linhas_invalidas)} rejeitadas")
    # Evidência para o README: latência maior que zero
    if linhas_validas:
        lat = [l["latencia_segundos"] for l in linhas_validas]
        print(f"Latência (s): mín {min(lat):.2f} | média {sum(lat)/len(lat):.2f} | máx {max(lat):.2f}")
    return len(response.received_messages)

## 7 — função de escuta

In [ ]:
def escutar(max_segundos=300, ocioso_segundos=45):
    """NOVO: modo escuta. Fica puxando mensagens, sem desistir na primeira fila vazia.
    - Para sozinho depois de `ocioso_segundos` sem mensagens novas (desde que já tenha recebido algo);
    - ou ao atingir `max_segundos` no total (trava de segurança).
    USO: rode esta célula ANTES do producer; ela espera. Depois inicie o producer em outra aba."""
    inicio = ultima_msg = time.time()
    recebeu_algo = False
    total = 0
    print(f"Escutando {subscription_id} (para após {ocioso_segundos}s sem mensagens ou em {max_segundos}s)...")
    while time.time() - inicio < max_segundos:
        n = consumir_micro_lotes()
        if n > 0:
            recebeu_algo, ultima_msg = True, time.time()
            total += n
        elif recebeu_algo and time.time() - ultima_msg >= ocioso_segundos:
            break
    print(f"Fim da escuta. Mensagens recebidas no total: {total}")

## 8 — configuração do producer

In [ ]:
import json
import random
import time
import uuid
from datetime import datetime, timezone   # timezone: o horário precisa ter fuso (UTC) para a latência funcionar

from google.cloud import pubsub_v1

project_id = "project-ce782257-1c9c-4b6f-87a"
# CORREÇÃO: antes era "topico-alfabetizacao " (com espaço no fim), e o Pub/Sub não achava o tópico
topic_id = "topico-alfabetizacao"

publisher = pubsub_v1.PublisherClient()
# monta o caminho completo do tópico: projects/<projeto>/topics/<tópico>
topic_path = publisher.topic_path(project_id, topic_id)

## 9 — evento válido

In [ ]:
# Limite oficial do INEP: a partir de 743 pontos o aluno está alfabetizado.
LIMIAR_ALFABETIZADO = 743


def gerar_evento():
    # Sorteia a proficiência na faixa real do microdado (cerca de 578 a 904).
    proficiencia = round(random.uniform(580.0, 900.0), 2)
    # CORREÇÃO: antes `alfabetizado` era sorteado à parte e podia sair "Sim" com 600 pontos.
    # Agora é DERIVADO da proficiência, como na fonte oficial. Texto "1"/"0", como na Silver.
    alfabetizado = "1" if proficiencia >= LIMIAR_ALFABETIZADO else "0"

    return {
        # --- controle do streaming (exigidos pelo enunciado) ---
        "id_evento": str(uuid.uuid4()),                          # identificador único: base da deduplicação
        # CORREÇÃO: utcnow() gera horário SEM fuso; o consumer usa horário COM fuso e a subtração falharia
        "horario_producao": datetime.now(timezone.utc).isoformat(),
        # CORREÇÃO: era "_source". Na Bronze, `_source` significa "tabela de origem";
        # aqui o que queremos dizer é "dado simulado", por isso a coluna se chama `origem`.
        "origem": "simulado",

        # --- colunas da silver_tc.alunos (códigos de texto, como na Silver) ---
        "ano": 2026,                       # INT64; 2026 cabe na partição da Silver (2023 a 2031)
        "id_municipio": "3550308",         # São Paulo; 7 dígitos (padrão IBGE)
        "cod_uf": "35",                    # texto, 2 primeiros dígitos do município
        "sigla_uf": "SP",
        "id_escola": str(random.randint(10000, 99999)),
        "id_aluno": str(random.randint(1000000, 9999999)),
        "caderno": str(random.randint(1, 21)),      # a fonte tem cadernos de 1 a 21
        "serie": "2",                      # texto
        "serie_descricao": "2º ano do Ensino Fundamental",
        "rede": "3",                       # texto; 3 = Municipal
        "rede_descricao": "Municipal",
        "presenca": "1",                   # texto; 1 = Presente
        "presenca_descricao": "Presente",
        "preenchimento_caderno": "1",      # texto; 1 = Prova preenchida
        "preenchimento_caderno_descricao": "Prova preenchida",
        "alfabetizado": alfabetizado,
        "alfabetizado_descricao": "Sim" if alfabetizado == "1" else "Não",   # coerente com o código
        # CORREÇÃO: era o texto "true". Na Silver `avaliado` é booleano.
        "avaliado": True,
        "proficiencia": round(proficiencia, 2),
        "peso_aluno": round(random.uniform(0.5, 1.5), 2),   # peso amostral positivo
    }

## 10 — evento inválido e mensagem quebrada

In [ ]:
def gerar_evento_rejeitado():
    """Evento inválido de propósito: sem id_municipio.
    CORREÇÃO: esta função estava declarada sem corpo, o que é erro de sintaxe e impedia a célula de rodar.
    Tem id_evento novo, então não é confundido com duplicado: o consumer deve rejeitá-lo
    e gravá-lo em ops_tc.eventos_rejeitados com o motivo."""
    evento = gerar_evento()
    del evento["id_municipio"]
    return evento


# JSON cortado no meio, para provar que o consumer não quebra com mensagem ilegível.
MENSAGEM_QUEBRADA = b'{"id_evento": "json-quebrado", '

## 11 — publicação

In [ ]:
def publicar(evento=None, bruto=None):
    """Publica uma mensagem. O Pub/Sub só aceita bytes: o dict vira JSON e depois bytes UTF-8.
    `bruto` permite enviar bytes já prontos (usado na mensagem quebrada)."""
    dados = bruto if bruto is not None else json.dumps(evento).encode("utf-8")
    # .result() espera a confirmação do Pub/Sub e levanta erro se a publicação falhar (billing, API, permissão)
    return publisher.publish(topic_path, data=dados).result()


def iniciar_simulador(qtd_validos=5, enviar_duplicado=True, enviar_invalido=True,
                      enviar_quebrado=False, pausa=2):
    print(f"Iniciando envio para o tópico: {topic_path}")
    ultimo = None

    # 1) eventos válidos, espaçados para simular chegada em tempo real
    for i in range(qtd_validos):
        ultimo = gerar_evento()
        id_pubsub = publicar(ultimo)
        print(f"[{i+1}/{qtd_validos}] válido enviado | id_evento={ultimo['id_evento']} | pubsub={id_pubsub}")
        time.sleep(pausa)

    # 2) duplicado: reenvia o ÚLTIMO evento, com o MESMO id_evento.
    # Prova o "o mesmo evento enviado 2 vezes aparece 1 vez" do roteiro
    # (o Pub/Sub entrega "pelo menos uma vez", então duplicata é esperada).
    if enviar_duplicado and ultimo:
        publicar(ultimo)
        print(f"duplicado enviado | id_evento={ultimo['id_evento']}")

    # 3) inválido: deve aparecer em ops_tc.eventos_rejeitados
    if enviar_invalido:
        invalido = gerar_evento_rejeitado()
        publicar(invalido)
        print(f"inválido enviado (sem id_municipio) | id_evento={invalido['id_evento']}")

    # 4) JSON quebrado (opcional): testa a rejeição por "JSON inválido"
    if enviar_quebrado:
        publicar(bruto=MENSAGEM_QUEBRADA)
        print("mensagem com JSON quebrado enviada")

    # resumo do que o consumer deve mostrar, para conferir de cabeça
    print("Fim. Esperado no consumer: "
          f"{qtd_validos} válidos, 1 duplicado ignorado, {int(enviar_invalido) + int(enviar_quebrado)} rejeitado(s).")

## 12 — TESTE: consumer escutando em thread + producer publicando

O consumer sobe primeiro e espera. Depois o producer envia 5 válidos, 1 duplicado e 1 inválido. A célula só termina quando o consumer para (20 s sem mensagens novas). Os prints do consumer aparecem nesta mesma saída.

In [ ]:
import threading

# 1) consumer escutando em segundo plano
consumidor = threading.Thread(target=escutar, kwargs=dict(max_segundos=180, ocioso_segundos=20))
consumidor.start()
time.sleep(3)   # dá tempo do consumer já estar esperando mensagens

# 2) producer publica enquanto o consumer escuta
iniciar_simulador(qtd_validos=5)

# 3) espera o consumer terminar (para sozinho após 20 s sem mensagens)
consumidor.join()
print('Teste concluído.')

## 13 — conferência no BigQuery (opcional, tabelas pequenas)

In [ ]:
# Descomente para tirar o print que vai para o README: latência e motivo da rejeição.
# print(bq_client.query(f"SELECT id_evento, origem, latencia_segundos FROM `{tabela_eventos}` ORDER BY horario_recebimento DESC LIMIT 20").to_dataframe())
# print(bq_client.query(f"SELECT id_evento, motivo_rejeicao FROM `{tabela_rejeitados}` ORDER BY horario_recebimento DESC LIMIT 20").to_dataframe())